In [2]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import LabelEncoder


def iqr_clip(df, columns, factor=1.5):
    df = df.copy()
    for col in columns:
        Q1 = df[col].quantile(0.25)
        Q3 = df[col].quantile(0.75)
        IQR = Q3 - Q1
        lower = Q1 - factor * IQR
        upper = Q3 + factor * IQR
        df[col] = df[col].clip(lower, upper)
    return df


def add_block_id(df):
    df = df.copy()
    class_change = (df['Class'] != df['Class'].shift(1)).astype(int)
    path_change = (df['PathOrder'] != df['PathOrder'].shift(1)).astype(int)
    df['block_id'] = (class_change | path_change).cumsum()
    return df


def split_blocks(df, train_frac=0.7, val_frac=0.1, meta_frac=0.1):
    df = df.copy()
    df['split'] = None
    for block, group in df.groupby('block_id'):
        n = len(group)
        n_train = int(n * train_frac)
        n_val = int(n * val_frac)
        n_meta = int(n * meta_frac)
        idx = group.index
        df.loc[idx[:n_train], 'split'] = 'convtran_train'          # ΔΕΝ χρησιμοποιείται εδώ
        df.loc[idx[n_train:n_train + n_val], 'split'] = 'convtran_val'  # ΔΕΝ χρησιμοποιείται εδώ
        df.loc[idx[n_train + n_val:n_train + n_val + n_meta], 'split'] = 'meta_train'
        df.loc[idx[n_train + n_val + n_meta:], 'split'] = 'test'
    return df


# ---------- Καθαρισμός ΟΛΟΥ του dataset, χωρίς φιλτράρισμα κλάσεων ----------
drivers_data = pd.read_csv('drivers_data.csv')

cols = pd.Series(drivers_data.columns)
for dupe in cols[cols.duplicated()].unique():
    cols[cols == dupe] = [f"{dupe}_{i}" if i != 0 else dupe for i in range(sum(cols == dupe))]
drivers_data.columns = cols

drivers_data['Class'] = drivers_data['Class'].astype(str).str.strip().str.upper()

to_drop = [
    'Filtered_Accelerator_Pedal_value', 'Glow_plug_control_request',
    'Inhibition_of_engine_fuel_cut_off', 'Fuel_Pressure',
    'Torque_scaling_factor(standardization)',
    'Target_engine_speed_used_in_lock-up_module', 'Converter_clutch', 'Clutch_operation_acknowledge',
    'TCU_requested_engine_RPM_increase', 'Indication_of_brake_switch_ON/OFF',
    'Standard_Torque_Ratio', 'Engine_in_fuel_cut_off',
    'TCU_requests_engine_torque_limit_(ETL)', 'Requested_spark_retard_angle_from_TCU',
    'Throttle_position_signal', 'Engine_speed', 'Flywheel_torque_(after_torque_interventions)',
    'Torque_converter_speed',
    'Wheel_velocity_front_left-hand', 'Wheel_velocity_front_right-hand',
    'Wheel_velocity_rear_left-hand',
    'Time(s)',
]
clean_df = drivers_data.drop(columns=to_drop, errors='ignore')

feature_cols = [c for c in clean_df.columns if c not in ['Class', 'PathOrder']]
clean_df = iqr_clip(clean_df, feature_cols, factor=1.5)

# ΠΡΟΣΟΧΗ: encoding A-J (10 κλάσεις), διαφορετικό από τα per-group encodings!
le = LabelEncoder()
clean_df['Class'] = le.fit_transform(clean_df['Class'])
print(f"Mapping (10 κλάσεις): {dict(zip(le.classes_, le.transform(le.classes_)))}")

clean_df = add_block_id(clean_df)
clean_df = split_blocks(clean_df, train_frac=0.7, val_frac=0.1, meta_frac=0.1)

print(clean_df.groupby(['Class', 'split']).size().unstack(fill_value=0))

clean_df.to_csv('cleaned_full_split.csv', index=False)

Mapping (10 κλάσεις): {'A': 0, 'B': 1, 'C': 2, 'D': 3, 'E': 4, 'F': 5, 'G': 6, 'H': 7, 'I': 8, 'J': 9}
split  convtran_train  convtran_val  meta_train  test
Class                                                
0                5067           723         723   727
1                9003          1284        1284  1293
2                5249           749         749   753
3                9269          1323        1323  1329
4                5904           843         843   846
5                7707          1101        1101  1103
6                5244           748         748   752
7                6915           987         987   991
8                5464           779         779   786
9                6231           889         889   895
